In [20]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
import os
import random
import scipy.sparse as sp
import torch
import torch.nn as nn
import argparse

# 1. 학습 설정

In [21]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [22]:
print(torch.cuda.is_available())

True


# 2. 데이터 전처리

In [23]:
class MakeGraphDataSet():
    def __init__(self, config):
        self.config = config
        # .dat 파일 불러오기
        self.df = pd.read_csv(os.path.join(self.config.data_path, 'ratings.dat'), delimiter='::', engine='python', names=['userId', 'movieId', 'rating', 'timestamp'], encoding='ISO-8859-1')
        
        self.item_encoder, self.item_decoder = self.generate_encoder_decoder('movieId')
        self.user_encoder, self.user_decoder = self.generate_encoder_decoder('userId')
        self.num_item, self.num_user = len(self.item_encoder), len(self.user_encoder)

        self.df['item_idx'] = self.df['movieId'].apply(lambda x : self.item_encoder[x])
        self.df['user_idx'] = self.df['userId'].apply(lambda x : self.user_encoder[x])
        
        self.exist_users = [i for i in range(self.num_user)]
        self.exist_items = [i for i in range(self.num_item)]
        self.user_train, self.user_valid = self.generate_sequence_data()
        self.R_train, self.R_valid, self.R_total = self.generate_dok_matrix()
        self.ngcf_adj_matrix = self.generate_ngcf_adj_matrix()
        self.n_train = len(self.R_train)
        self.batch_size = self.config.batch_size


    def generate_encoder_decoder(self, col : str) -> dict:


        encoder = {}
        decoder = {}
        ids = self.df[col].unique()

        for idx, _id in enumerate(ids):
            encoder[_id] = idx
            decoder[idx] = _id

        return encoder, decoder
    
    def generate_sequence_data(self) -> dict:

        users = defaultdict(list)
        user_train = {}
        user_valid = {}
        for user, item, time in zip(self.df['user_idx'], self.df['item_idx'], self.df['timestamp']):
            users[user].append(item)
        
        for user in users:
            np.random.seed(self.config.seed)

            user_total = users[user]
            valid = np.random.choice(user_total, size = self.config.valid_samples, replace = False).tolist()
            train = list(set(user_total) - set(valid))

            user_train[user] = train
            user_valid[user] = valid # valid_samples 개수 만큼 검증에 활용 (현재 Task와 가장 유사하게)

        return user_train, user_valid
    
    def generate_dok_matrix(self):
        R_train = sp.dok_matrix((self.num_user, self.num_item), dtype=np.float32)
        R_valid = sp.dok_matrix((self.num_user, self.num_item), dtype=np.float32)
        R_total = sp.dok_matrix((self.num_user, self.num_item), dtype=np.float32)
        for idx, row in self.df.iterrows():
            user_idx, item_idx, rating = row['user_idx'], row['item_idx'], row['rating']
            if item_idx in self.user_train[user_idx]:
                R_train[user_idx, item_idx] = rating
                R_total[user_idx, item_idx] = rating
            elif item_idx in self.user_valid[user_idx]:
                R_valid[user_idx, item_idx] = rating
                R_total[user_idx, item_idx] = rating
        return R_train, R_valid, R_total

    def generate_ngcf_adj_matrix(self):
        adj_mat = sp.dok_matrix((self.num_user + self.num_item, self.num_user + self.num_item), dtype=np.float32)
        adj_mat = adj_mat.tolil() # to_list
        R = self.R_train.tolil()

        adj_mat[:self.num_user, self.num_user:] = R
        adj_mat[self.num_user:, :self.num_user] = R.T
        adj_mat = adj_mat.todok() # to_dok_matrix

        def normalized_adj_single(adj):
            rowsum = np.array(adj.sum(1))
            d_inv = np.power(rowsum, -.5).flatten()  
            d_inv[np.isinf(d_inv)] = 0.
            d_mat_inv = sp.diags(d_inv)
            norm_adj = d_mat_inv.dot(adj).dot(d_mat_inv)

            return norm_adj.tocoo()

        ngcf_adj_matrix = normalized_adj_single(adj_mat)
        return ngcf_adj_matrix.tocsr()

    def sampling(self):
        users = random.sample(self.exist_users, self.config.batch_size)

        def sample_pos_items_for_u(u, num):
            pos_items = self.user_train[u]
            pos_batch = random.sample(pos_items, num)
            return pos_batch
        
        def sample_neg_items_for_u(u, num):
            neg_items = list(set(self.exist_items) - set(self.user_train[u]))
            neg_batch = random.sample(neg_items, num)
            return neg_batch
        
        pos_items, neg_items = [], []
        for user in users:
            pos_items += sample_pos_items_for_u(user, 1)
            neg_items += sample_neg_items_for_u(user, 1)
        
        return users, pos_items, neg_items

    def get_train_valid_data(self):
        return self.user_train, self.user_valid

    def get_R_data(self):
        return self.R_train, self.R_valid, self.R_total

    def get_ngcf_adj_matrix_data(self):
        return self.ngcf_adj_matrix

# 3. 모델

In [24]:
class LightGCN(nn.Module):
    def __init__(self, n_users, n_items, emb_dim, n_layers, reg, node_dropout, adj_mtx):
        super().__init__()

        # initialize Class attributes
        self.n_users = n_users
        self.n_items = n_items
        self.emb_dim = emb_dim
        self.l = adj_mtx
        self.graph = self._convert_sp_mat_to_sp_tensor(self.l)

        self.reg = reg
        self.n_layers = n_layers
        self.node_dropout = node_dropout

        # Initialize weights
        self.weight_dict = self._init_weights()
        print("Weights initialized.")

    # initialize weights
    def _init_weights(self):
        print("Initializing weights...")
        weight_dict = nn.ParameterDict()

        initializer = torch.nn.init.xavier_uniform_
        
        weight_dict['user_embedding'] = nn.Parameter(initializer(torch.empty(self.n_users, self.emb_dim).to(device)))
        weight_dict['item_embedding'] = nn.Parameter(initializer(torch.empty(self.n_items, self.emb_dim).to(device)))
           
        return weight_dict

    # convert sparse matrix into sparse PyTorch tensor
    def _convert_sp_mat_to_sp_tensor(self, X):

        coo = X.tocoo().astype(np.float32)
        i = torch.LongTensor(np.mat([coo.row, coo.col]))
        v = torch.FloatTensor(coo.data)
        res = torch.sparse.FloatTensor(i, v, coo.shape).to(device)
        return res

    # apply node_dropout
    def _droupout_sparse(self, X):

        node_dropout_mask = ((self.node_dropout) + torch.rand(X._nnz())).floor().bool().to(device)
        i = X.coalesce().indices()
        v = X.coalesce()._values()
        i[:,node_dropout_mask] = 0
        v[node_dropout_mask] = 0
        X_dropout = torch.sparse.FloatTensor(i, v, X.shape).to(X.device)

        return  X_dropout.mul(1/(1-self.node_dropout))

    def forward(self, u, i):
        # apply drop-out mask
        graph = self._droupout_sparse(self.graph) if self.node_dropout > 0 else self.graph
        ego_embeddings = torch.cat([self.weight_dict['user_embedding'], self.weight_dict['item_embedding']], 0)
        final_embeddings = [ego_embeddings]

        for k in range(self.n_layers):
            ego_embeddings = torch.sparse.mm(graph, final_embeddings[k])
            final_embeddings.append(ego_embeddings)                                       

        final_embeddings = torch.stack(final_embeddings, dim=1)
        final_embeddings = torch.mean(final_embeddings, dim=1)
        
        u_final_embeddings, i_final_embeddings = final_embeddings.split([self.n_users, self.n_items], 0)

        self.u_final_embeddings = nn.Parameter(u_final_embeddings)
        self.i_final_embeddings = nn.Parameter(i_final_embeddings)
        
        # loss 계산
        u_emb = u_final_embeddings[u]
        i_emb = i_final_embeddings[i]
        prediction = torch.sum(torch.mul(u_emb, i_emb), dim=1)
        return prediction

# 4. 학습 함수

In [25]:
def train(model, make_graph_data_set, optimizer, n_batch, R_train):
    model.train()
    total_loss = 0
    for step in range(1, n_batch + 1):
        user, pos, _ = make_graph_data_set.sampling() # neg sampling 불필요
        optimizer.zero_grad()
        preds = model(user, pos)
        criterion = torch.nn.MSELoss().to(device)
        labels = get_ratings_for_batch(user, pos, R_train) # 실제 평점 가져오기
        loss = criterion(preds, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / n_batch

# 평점 가져오기
def get_ratings_for_batch(users, items, R):
    ratings = [R[u, i] for u, i in zip(users, items)]
    return torch.Tensor(ratings).to(device)

def split_matrix(X, n_splits=10):
    splits = []
    chunk_size = X.shape[0] // n_splits
    for i in range(n_splits):
        start = i * chunk_size
        end = X.shape[0] if i == n_splits - 1 else (i + 1) * chunk_size
        splits.append(X[start:end])
    return splits

def compute_ndcg_k(pred_items, test_items, test_indices, k):
    
    r = (test_items * pred_items).gather(1, test_indices)
    f = torch.from_numpy(np.log2(np.arange(2, k+2))).float().to(device)
    
    dcg = (r[:, :k]/f).sum(1)                                               
    dcg_max = (torch.sort(r, dim=1, descending=True)[0][:, :k]/f).sum(1)   
    ndcg = dcg/dcg_max                                                     
    
    ndcg[torch.isnan(ndcg)] = 0
    return ndcg

def evaluate(u_emb, i_emb, Rtr, Rte, k = 10):

    # split matrices
    ue_splits = split_matrix(u_emb)
    tr_splits = split_matrix(Rtr)
    te_splits = split_matrix(Rte)

    recall_k, ndcg_k= [], []
    # compute results for split matrices
    for ue_f, tr_f, te_f in zip(ue_splits, tr_splits, te_splits):

        scores = torch.mm(ue_f, i_emb.t())

        test_items = torch.from_numpy(te_f.todense()).float().to(device)
        non_train_items = torch.from_numpy(1-(tr_f.todense())).float().to(device)
        scores = scores * non_train_items

        _, test_indices = torch.topk(scores, dim=1, k=k)
        
        pred_items = torch.zeros_like(scores).float()
        pred_items.scatter_(dim=1, index=test_indices, src=torch.ones_like(test_indices).float().to(device))

        topk_preds = torch.zeros_like(scores).float()
        topk_preds.scatter_(dim=1, index=test_indices[:, :k], src=torch.ones_like(test_indices).float())
        
        TP = (test_items * topk_preds).sum(1)                      
        rec = TP/test_items.sum(1)
   
        ndcg = compute_ndcg_k(pred_items, test_items, test_indices, k)

        recall_k.append(rec)
        ndcg_k.append(ndcg)

    return torch.cat(ndcg_k).mean(), torch.cat(recall_k).mean()

# 5. 학습

In [26]:
class Args:
    def __init__(self):
        self.data_path = "./datasets"
        self.model_path = "./checkpoint_LGCN"
        self.model_name = "LightGCN_use_rating.pt"
        self.num_epochs = 50
        self.reg = 1e-5
        self.lr = 0.001
        self.emb_dim = 128
        self.n_layers = 3
        self.batch_size = 500
        self.node_dropout = 0.2
        self.valid_samples = 10
        self.seed = 22
        self.n_batch = 10

args = Args()

make_graph_data_set = MakeGraphDataSet(args)
ngcf_adj_matrix = make_graph_data_set.get_ngcf_adj_matrix_data()
R_train, R_valid, R_total = make_graph_data_set.get_R_data()

model = LightGCN(
    n_users=make_graph_data_set.num_user,
    n_items=make_graph_data_set.num_item,
    emb_dim=args.emb_dim,
    n_layers=args.n_layers,
    reg=args.reg,
    node_dropout=args.node_dropout,
    adj_mtx=ngcf_adj_matrix
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)

# 모델 저장 경로 체크 및 폴더 생성
if not os.path.exists(args.model_path):
    os.makedirs(args.model_path)

best_hit = 0
for epoch in range(1, args.num_epochs + 1):
    tbar = tqdm(range(1))
    for _ in tbar:
        train_loss = train(
            model = model, 
            make_graph_data_set = make_graph_data_set, 
            optimizer = optimizer,
            n_batch = args.n_batch,
            R_train = R_train
            )
        with torch.no_grad():
            ndcg, hit = evaluate(
                u_emb = model.u_final_embeddings.detach(), 
                i_emb = model.i_final_embeddings.detach(), 
                Rtr = R_train, 
                Rte = R_valid, 
                k = 10
                )

        if best_hit < hit:
            best_hit = hit
            torch.save(model.state_dict(), os.path.join(args.model_path, args.model_name))

        tbar.set_description(f'Epoch: {epoch:3d}| Train loss: {train_loss:.5f}| NDCG@10: {ndcg:.5f}| HIT@10: {hit:.5f}')

C:\Users\AI18\AppData\Local\Temp\ipykernel_12552\2665957791.py:81: RuntimeWarning: divide by zero encountered in power
  d_inv = np.power(rowsum, -.5).flatten()


Initializing weights...
Weights initialized.


Epoch:   1| Train loss: 15.00926| NDCG@10: 0.27537| HIT@10: 0.07762: 100%|██████████| 1/1 [00:10<00:00, 10.39s/it]
Epoch:   2| Train loss: 14.80493| NDCG@10: 0.28167| HIT@10: 0.07984: 100%|██████████| 1/1 [00:09<00:00,  9.92s/it]
Epoch:   3| Train loss: 14.64252| NDCG@10: 0.28264| HIT@10: 0.08088: 100%|██████████| 1/1 [00:09<00:00,  9.89s/it]
Epoch:   4| Train loss: 14.04527| NDCG@10: 0.28079| HIT@10: 0.08047: 100%|██████████| 1/1 [00:09<00:00,  9.60s/it]
Epoch:   5| Train loss: 13.16364| NDCG@10: 0.28275| HIT@10: 0.08123: 100%|██████████| 1/1 [00:09<00:00,  9.97s/it]
Epoch:   6| Train loss: 12.44882| NDCG@10: 0.28195| HIT@10: 0.08054: 100%|██████████| 1/1 [00:10<00:00, 10.10s/it]
Epoch:   7| Train loss: 11.06600| NDCG@10: 0.28376| HIT@10: 0.08070: 100%|██████████| 1/1 [00:09<00:00, 10.00s/it]
Epoch:   8| Train loss: 9.68859| NDCG@10: 0.28244| HIT@10: 0.08089: 100%|██████████| 1/1 [00:10<00:00, 10.15s/it]
Epoch:   9| Train loss: 8.50600| NDCG@10: 0.28176| HIT@10: 0.08056: 100%|████████

In [27]:
def recommend_top_k_movies_for_user(user_idx, model, make_graph_data_set, k=10):
    # 1. 이미 평가된 영화 제외
    watched_movies = set(make_graph_data_set.df[make_graph_data_set.df['user_idx'] == user_idx]['item_idx'].tolist())
    all_movies = set(make_graph_data_set.exist_items)
    unwatched_movies = list(all_movies - watched_movies)
    
    # 2. 점수 예측
    user_emb = model.u_final_embeddings[user_idx]
    unwatched_movie_embs = model.i_final_embeddings[unwatched_movies]
    scores = torch.matmul(user_emb, unwatched_movie_embs.t()).detach().cpu().numpy()

    # 3. 상위 k개 영화 선정
    top_k_idx = np.argsort(scores)[-k:][::-1]
    top_k_movie_idx = [unwatched_movies[i] for i in top_k_idx]
    top_k_movie_ids = [make_graph_data_set.item_decoder[idx] for idx in top_k_movie_idx]

    return top_k_movie_ids

# 사용 예시
model_path = os.path.join(args.model_path, args.model_name)
model.load_state_dict(torch.load(model_path))  # 모델 가중치 로드
model.eval()  # 추론 모드로 변경

user_idx = 0  # 추천을 받고자 하는 사용자 인덱스
recommended_movie_ids = recommend_top_k_movies_for_user(user_idx, model, make_graph_data_set)
print("Recommended movie IDs:", recommended_movie_ids)


Recommended movie IDs: [2858, 1196, 593, 1210, 2571, 110, 1198, 318, 858, 589]


In [28]:
def load_movies_data(filepath='./datasets/movies.dat'):
    # movies 데이터셋 로드
    movies = pd.read_csv(filepath, delimiter='::', engine='python', names=['MovieID', 'Title', 'Genres'], encoding='ISO-8859-1')
    # MovieID를 index로 설정
    movies.set_index('MovieID', inplace=True)
    return movies

def get_movie_titles_from_ids(movie_ids, movies_data):
    # 해당 MovieID에 대한 제목 가져오기
    return movies_data['Title'].loc[movie_ids].tolist()

# 사용 예시
model_path = os.path.join(args.model_path, args.model_name)
model.load_state_dict(torch.load(model_path))  # 모델 가중치 로드
model.eval()  # 추론 모드로 변경

user_idx = 0  # 추천을 받고자 하는 사용자 인덱스
recommended_movie_ids = recommend_top_k_movies_for_user(user_idx, model, make_graph_data_set)

# MovieID를 제목으로 변환
movies_data = load_movies_data()
top_10_titles = get_movie_titles_from_ids(recommended_movie_ids, movies_data)

print("Recommended movie titles:", top_10_titles)

Recommended movie titles: ['American Beauty (1999)', 'Star Wars: Episode V - The Empire Strikes Back (1980)', 'Silence of the Lambs, The (1991)', 'Star Wars: Episode VI - Return of the Jedi (1983)', 'Matrix, The (1999)', 'Braveheart (1995)', 'Raiders of the Lost Ark (1981)', 'Shawshank Redemption, The (1994)', 'Godfather, The (1972)', 'Terminator 2: Judgment Day (1991)']
